In [ ]:
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.Chem import rdDepictor
from collections import namedtuple
import re

def make_rect(match):
    cx = match.group(1)
    cy = match.group(2)
    rx = float(match.group(3))
    ry = float(match.group(4))
    atom_class = match.group(5)
    style_str = match.group(6)

    # 2. CALCULATE BOX
    width = rx * 1.5
    height = ry * 1.5
    x = float(cx) - (width / 2)
    y = float(cy) - (height / 2)

    return f"<rect x='{x}' y='{y}' width='{width}' height='{height}' rx='3' class='{atom_class}' style='{style_str}' />"

def hex_replace(match):
    full_style = match.group(0)
    # Find the hex code inside the style string
    hex8 = re.search(r'#([0-9A-Fa-f]{6})([0-9A-Fa-f]{2})', full_style)
    if hex8:
        color, alpha = hex8.groups()
        opacity = round(int(alpha, 16) / 255.0, 3)
        # Replace 8-digit with 6-digit
        fixed = full_style.replace(f"#{color}{alpha}", f"#{color}")
        # Add opacity attributes before the closing quote
        return fixed[:-1] + f";fill-opacity:{opacity};stroke-opacity:{opacity}'"
    return full_style


def replace_circles_with_rectangles(svg):
    pattern = r"<ellipse\s+cx='([^']+)'\s+cy='([^']+)'\s+rx='([^']+)'\s+ry='([^']+)'\s+class='([^']+)'\s+style='([^']+)'\s*/>"

    svg = re.sub(pattern, make_rect, svg)
    return re.sub(r"style='[^']+#([0-9A-Fa-f]{8})[^']*'", hex_replace, svg)


def DrawMorganBits(tpls, **kwargs):
  envs = []
  for tpl in tpls:
    if len(tpl) == 4:
      mol, bitId, bitInfo, whichExample = tpl
    else:
      mol, bitId, bitInfo = tpl
      whichExample = 0

    atomId, radius = bitInfo[bitId][whichExample]
    envs.append((mol, atomId, radius))
  return DrawMorganEnvs(envs, **kwargs)


# adapted from the function drawFPBits._drawFPBit() from the CheTo package
# original author Nadine Schneider
FingerprintEnv = namedtuple(
  'FingerprintEnv',
  ('submol', 'highlightAtoms', 'atomColors', 'highlightBonds', 'bondColors', 'highlightRadii'))


def _getMorganEnv(mol, atomId, radius, baseRad, aromaticColor, ringColor, centerColor, extraColor,
                  **kwargs):
  if not mol.GetNumConformers():
    rdDepictor.Compute2DCoords(mol)
  bitPath = Chem.FindAtomEnvironmentOfRadiusN(mol, radius, atomId)

  # get the atoms for highlighting
  atomsToUse = set((atomId, ))
  for b in bitPath:
    atomsToUse.add(mol.GetBondWithIdx(b).GetBeginAtomIdx())
    atomsToUse.add(mol.GetBondWithIdx(b).GetEndAtomIdx())

  #  enlarge the environment by one further bond
  enlargedEnv = set()
  for atom in atomsToUse:
    a = mol.GetAtomWithIdx(atom)
    for b in a.GetBonds():
      bidx = b.GetIdx()
      if bidx not in bitPath:
        enlargedEnv.add(bidx)
  enlargedEnv = list(enlargedEnv)
  enlargedEnv += bitPath

  # set the coordinates of the submol based on the coordinates of the original molecule
  amap = {}
  if enlargedEnv:
    submol = Chem.PathToSubmol(mol, enlargedEnv, atomMap=amap)
  else:
    # generate submol from fragments with no bonds
    submol = Chem.MolFromSmiles(Chem.MolFragmentToSmiles(mol, atomsToUse=atomsToUse))
  Chem.FastFindRings(submol)
  conf = Chem.Conformer(submol.GetNumAtoms())
  confOri = mol.GetConformer(0)
  for i1, i2 in amap.items():
    conf.SetAtomPosition(i2, confOri.GetAtomPosition(i1))
  submol.AddConformer(conf)

  for orig_idx, sub_idx in amap.items():
        sub_atom = submol.GetAtomWithIdx(sub_idx)

        # --- MODIFICATION START ---
        # Check if the atom belongs to the "Core" bit
        if orig_idx in atomsToUse:
            orig_atom = mol.GetAtomWithIdx(orig_idx)
            h_count = orig_atom.GetTotalNumHs()
            degree = orig_atom.GetDegree()
            sub_atom.SetProp("atomNote", f"H:{h_count},D:{degree}")
        else:
            # For atoms outside the core, ensure no label is set
            if sub_atom.HasProp("atomNote"):
                sub_atom.ClearProp("atomNote")
            # Convert to Wildcard as before
            sub_atom.SetAtomicNum(0)
            sub_atom.SetIsAromatic(False)
        # --- MODIFICATION END ---

  envSubmol = []
  for b in bitPath:
      beginAtom = amap[mol.GetBondWithIdx(b).GetBeginAtomIdx()]
      endAtom = amap[mol.GetBondWithIdx(b).GetEndAtomIdx()]
      envSubmol.append(submol.GetBondBetweenAtoms(beginAtom, endAtom).GetIdx())

  # color all atoms of the submol in gray which are not part of the bit
  # highlight atoms which are in rings
  atomcolors, bondcolors = {}, {}
  highlightAtoms, highlightBonds = [], []
  highlightRadii = {}
  for aidx in amap.keys():
    if aidx in atomsToUse:
      color = None
      if centerColor and aidx == atomId:
        color = centerColor
      if color is not None:
        atomcolors[amap[aidx]] = color
        highlightAtoms.append(amap[aidx])
        highlightRadii[amap[aidx]] = baseRad
    else:
      #drawopt.atomLabels[amap[aidx]] = '*'
      submol.GetAtomWithIdx(amap[aidx]).SetAtomicNum(0)
      submol.GetAtomWithIdx(amap[aidx]).UpdatePropertyCache()
  color = extraColor
  for bid in submol.GetBonds():
    bidx = bid.GetIdx()
    if bidx not in envSubmol:
      bondcolors[bidx] = color
      bid.SetBondType(Chem.BondType.OTHER)
      bid.SetIsAromatic(False)

      bondcolors[bidx] = color
      highlightBonds.append(bidx)

  return FingerprintEnv(submol, highlightAtoms, atomcolors, highlightBonds, bondcolors,
                        highlightRadii)


def DrawMorganEnvs(envs, molsPerRow=3, subImgSize=(150, 150), baseRad=0.4, useSVG=True,
                   aromaticColor=(0.9, 0.9, 0.2), ringColor=(0.8, 0.8, 0.8),
                   centerColor=(0.788, 0, 0.329, 0.4), extraColor=(0.9, 0.9, 0.9), legends=None,
                   drawOptions=None, **kwargs):
  submols = []
  highlightAtoms = []
  atomColors = []
  highlightBonds = []
  bondColors = []
  highlightRadii = []
  for mol, atomId, radius in envs:
    menv = _getMorganEnv(mol, atomId, radius, baseRad, aromaticColor, ringColor, centerColor,
                         extraColor, **kwargs)
    submols.append(menv.submol)
    highlightAtoms.append(menv.highlightAtoms)
    atomColors.append(menv.atomColors)
    highlightBonds.append(menv.highlightBonds)
    bondColors.append(menv.bondColors)
    highlightRadii.append(menv.highlightRadii)

  if legends is None:
    legends = [''] * len(envs)

  nRows = len(envs) // molsPerRow
  if len(envs) % molsPerRow:
    nRows += 1

  fullSize = (molsPerRow * subImgSize[0], nRows * subImgSize[1])
  # Drawing
  if useSVG:
    drawer = rdMolDraw2D.MolDraw2DSVG(fullSize[0], fullSize[1], subImgSize[0], subImgSize[1])
  else:
    drawer = rdMolDraw2D.MolDraw2DCairo(fullSize[0], fullSize[1], subImgSize[0], subImgSize[1])

  if drawOptions is None:
    drawOptions = drawer.drawOptions()
  drawOptions.prepareMolsBeforeDrawing = False
  drawOptions.continuousHighlight = False
  drawOptions.includeMetadata = False
  drawer.SetDrawOptions(drawOptions)
  drawer.DrawMolecules(submols, legends=legends, highlightAtoms=highlightAtoms,
                       highlightAtomColors=atomColors, highlightBonds=highlightBonds,
                       highlightBondColors=bondColors, highlightAtomRadii=highlightRadii, **kwargs)
  drawer.FinishDrawing()
  svg = drawer.GetDrawingText()

  return replace_circles_with_rectangles(svg)

In [ ]:
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.Chem import rdDepictor
from collections import namedtuple
import re


def make_rect(match):
    cx = match.group(1)
    cy = match.group(2)
    rx = float(match.group(3))
    ry = float(match.group(4))
    atom_class = match.group(5)
    style_str = match.group(6)

    # 2. CALCULATE BOX
    width = rx * 1.5
    height = ry * 1.5
    x = float(cx) - (width / 2)
    y = float(cy) - (height / 2)

    return f"<rect x='{x}' y='{y}' width='{width}' height='{height}' rx='3' class='{atom_class}' style='{style_str}' />"


def hex_replace(match):
    full_style = match.group(0)
    # Find the hex code inside the style string
    hex8 = re.search(r'#([0-9A-Fa-f]{6})([0-9A-Fa-f]{2})', full_style)
    if hex8:
        color, alpha = hex8.groups()
        opacity = round(int(alpha, 16) / 255.0, 3)
        # Replace 8-digit with 6-digit
        fixed = full_style.replace(f"#{color}{alpha}", f"#{color}")
        # Add opacity attributes before the closing quote
        return fixed[:-1] + f";fill-opacity:{opacity};stroke-opacity:{opacity}'"
    return full_style


def replace_circles_with_rectangles(svg):
    pattern = r"<ellipse\s+cx='([^']+)'\s+cy='([^']+)'\s+rx='([^']+)'\s+ry='([^']+)'\s+class='([^']+)'\s+style='([^']+)'\s*/>"

    svg = re.sub(pattern, make_rect, svg)
    return re.sub(r"style='[^']+#([0-9A-Fa-f]{8})[^']*'", hex_replace, svg)


def DrawMorganBits(tpls, **kwargs):
    envs = []
    for tpl in tpls:
        if len(tpl) == 4:
            mol, bitId, bitInfo, whichExample = tpl
        else:
            mol, bitId, bitInfo = tpl
            whichExample = 0

        atomId, radius = bitInfo[bitId][whichExample]
        envs.append((mol, atomId, radius))
    return DrawMorganEnvs(envs, **kwargs)


# adapted from the function drawFPBits._drawFPBit() from the CheTo package
# original author Nadine Schneider
FingerprintEnv = namedtuple(
    'FingerprintEnv',
    ('submol', 'highlightAtoms', 'atomColors', 'highlightBonds', 'bondColors', 'highlightRadii'))


def _getMorganEnv(mol, atomId, radius, baseRad, aromaticColor, ringColor, centerColor, extraColor,
                  **kwargs):
    if not mol.GetNumConformers():
        rdDepictor.Compute2DCoords(mol)
    bitPath = Chem.FindAtomEnvironmentOfRadiusN(mol, radius, atomId)

    # get the atoms for highlighting
    atomsToUse = set((atomId,))
    for b in bitPath:
        atomsToUse.add(mol.GetBondWithIdx(b).GetBeginAtomIdx())
        atomsToUse.add(mol.GetBondWithIdx(b).GetEndAtomIdx())

    #  enlarge the environment by one further bond
    enlargedEnv = set()
    for atom in atomsToUse:
        a = mol.GetAtomWithIdx(atom)
        for b in a.GetBonds():
            bidx = b.GetIdx()
            if bidx not in bitPath:
                enlargedEnv.add(bidx)
    enlargedEnv = list(enlargedEnv)
    enlargedEnv += bitPath

    # set the coordinates of the submol based on the coordinates of the original molecule
    amap = {}
    if enlargedEnv:
        submol = Chem.PathToSubmol(mol, enlargedEnv, atomMap=amap)
    else:
        # generate submol from fragments with no bonds
        submol = Chem.MolFromSmiles(Chem.MolFragmentToSmiles(mol, atomsToUse=atomsToUse))
    Chem.FastFindRings(submol)
    conf = Chem.Conformer(submol.GetNumAtoms())
    confOri = mol.GetConformer(0)
    for i1, i2 in amap.items():
        conf.SetAtomPosition(i2, confOri.GetAtomPosition(i1))
    submol.AddConformer(conf)

    for orig_idx, sub_idx in amap.items():
        sub_atom = submol.GetAtomWithIdx(sub_idx)

        # --- MODIFICATION START ---
        # Check if the atom belongs to the "Core" bit
        if orig_idx in atomsToUse:
            orig_atom = mol.GetAtomWithIdx(orig_idx)
            h_count = orig_atom.GetTotalNumHs()
            degree = orig_atom.GetDegree()
            sub_atom.SetProp("atomNote", f"H:{h_count},D:{degree}")
        else:
            # For atoms outside the core, ensure no label is set
            if sub_atom.HasProp("atomNote"):
                sub_atom.ClearProp("atomNote")
            # Convert to Wildcard as before
            sub_atom.SetAtomicNum(0)
            sub_atom.SetIsAromatic(False)
        # --- MODIFICATION END ---

    envSubmol = []
    for b in bitPath:
        beginAtom = amap[mol.GetBondWithIdx(b).GetBeginAtomIdx()]
        endAtom = amap[mol.GetBondWithIdx(b).GetEndAtomIdx()]
        envSubmol.append(submol.GetBondBetweenAtoms(beginAtom, endAtom).GetIdx())

    # color all atoms of the submol in gray which are not part of the bit
    # highlight atoms which are in rings
    atomcolors, bondcolors = {}, {}
    highlightAtoms, highlightBonds = [], []
    highlightRadii = {}
    for aidx in amap.keys():
        if aidx in atomsToUse:
            color = None
            if centerColor and aidx == atomId:
                color = centerColor
            if color is not None:
                atomcolors[amap[aidx]] = color
                highlightAtoms.append(amap[aidx])
                highlightRadii[amap[aidx]] = baseRad
        else:
            #drawopt.atomLabels[amap[aidx]] = '*'
            submol.GetAtomWithIdx(amap[aidx]).SetAtomicNum(0)
            submol.GetAtomWithIdx(amap[aidx]).UpdatePropertyCache()
    color = extraColor
    for bid in submol.GetBonds():
        bidx = bid.GetIdx()
        if bidx not in envSubmol:
            bondcolors[bidx] = color
            bid.SetBondType(Chem.BondType.OTHER)
            bid.SetIsAromatic(False)

            bondcolors[bidx] = color
            highlightBonds.append(bidx)

    return FingerprintEnv(submol, highlightAtoms, atomcolors, highlightBonds, bondcolors,
                          highlightRadii)


def DrawMorganEnvs(envs, molsPerRow=3, subImgSize=(150, 150), baseRad=0.4, useSVG=True,
                   aromaticColor=(0.9, 0.9, 0.2), ringColor=(0.8, 0.8, 0.8),
                   centerColor=(0.788, 0, 0.329, 0.4), extraColor=(0.9, 0.9, 0.9), legends=None,
                   drawOptions=None, **kwargs):
    submols = []
    highlightAtoms = []
    atomColors = []
    highlightBonds = []
    bondColors = []
    highlightRadii = []
    for mol, atomId, radius in envs:
        menv = _getMorganEnv(mol, atomId, radius, baseRad, aromaticColor, ringColor, centerColor,
                             extraColor, **kwargs)
        submols.append(menv.submol)
        highlightAtoms.append(menv.highlightAtoms)
        atomColors.append(menv.atomColors)
        highlightBonds.append(menv.highlightBonds)
        bondColors.append(menv.bondColors)
        highlightRadii.append(menv.highlightRadii)

    if legends is None:
        legends = [''] * len(envs)

    nRows = len(envs) // molsPerRow
    if len(envs) % molsPerRow:
        nRows += 1

    fullSize = (molsPerRow * subImgSize[0], nRows * subImgSize[1])
    # Drawing
    if useSVG:
        drawer = rdMolDraw2D.MolDraw2DSVG(fullSize[0], fullSize[1], subImgSize[0], subImgSize[1])
    else:
        drawer = rdMolDraw2D.MolDraw2DCairo(fullSize[0], fullSize[1], subImgSize[0], subImgSize[1])

    if drawOptions is None:
        drawOptions = drawer.drawOptions()
    drawOptions.prepareMolsBeforeDrawing = False
    drawOptions.continuousHighlight = False
    drawOptions.includeMetadata = False
    drawer.SetDrawOptions(drawOptions)
    drawer.DrawMolecules(submols, legends=legends, highlightAtoms=highlightAtoms,
                         highlightAtomColors=atomColors, highlightBonds=highlightBonds,
                         highlightBondColors=bondColors, highlightAtomRadii=highlightRadii, **kwargs)
    drawer.FinishDrawing()
    svg = drawer.GetDrawingText()

    return replace_circles_with_rectangles(svg)

def summarize_bits(smiles_list, target_bit, radius=2, n_bits=2048, use_chiral=True):
    """
    Groups molecules by their core bit structure and visualizes unique cores.
    """
    unique_cores = {}

    for m_idx, smiles in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smiles)
        if not mol: continue

        Chem.AssignStereochemistry(mol, force=True, cleanIt=True)
        bit_info = {}
        AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits, bitInfo=bit_info,
                                              useChirality=use_chiral)

        if target_bit in bit_info:
            # Get the core substructure for aggregation
            atomId, rad = bit_info[target_bit][0]
            bitPath = Chem.FindAtomEnvironmentOfRadiusN(mol, rad, atomId)

            # Extract just the core (black part) to generate a grouping key
            core_submol = Chem.PathToSubmol(mol, bitPath)
            core_key = Chem.MolToSmiles(core_submol, isomericSmiles=use_chiral)

            if core_key not in unique_cores:
                unique_cores[core_key] = {
                    'mol': mol,
                    'atomId': atomId,
                    'rad': rad,
                    'mol_indices': set()
                }
            unique_cores[core_key]['mol_indices'].add(m_idx)

    # Prepare data for DrawMorganEnvs
    envs_to_draw = []
    legends = []
    for core_key, data in unique_cores.items():
        envs_to_draw.append((data['mol'], data['atomId'], data['rad']))
        legends.append(f"Found in {len(data['mol_indices'])} molecules")

    if not envs_to_draw:
        return "No occurrences found."

    img = DrawMorganEnvs(envs_to_draw, legends=legends, subImgSize=(300, 300))
    return img

def visualize_on_molecule(smiles: str, i, target_bit: int, radius: int = 2, n_bits: int = 2048,
                          use_chiral: bool = True):
    """
    Visualizes a specific Bit ID highlighted on the molecule.
    """
    # 1. Prepare Molecule
    mol = Chem.MolFromSmiles(smiles)
    if not mol:
        print("Invalid SMILES")
        return

    # Assign Stereochemistry so RDKit knows about R/S labels
    Chem.AssignStereochemistry(mol, force=True, cleanIt=True)

    # 2. Calculate Fingerprint with Chirality
    bit_info = {}
    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol, radius=radius, nBits=n_bits, bitInfo=bit_info, useChirality=use_chiral
    )

    if target_bit not in bit_info:
        return None

    instances = bit_info[target_bit]

    # 1. Track highlights and specific colors
    highlight_atoms = []
    highlight_bonds = []
    atom_colors = {}  # Dictionary to map atom index to color (R, G, B)

    # Define our Blue color for the central atoms
    red_color = (0.204, 1, 0.941, 0.4)
    blue_color = (0.788, 0, 0.329, 0.4)

    highlight_atoms = []
    highlight_bonds = []
    atom_colors = {}
    bond_colors = {}

    instances = bit_info[target_bit]

    for atom_idx, rad in instances:
        # Mark center as Blue
        atom_colors[atom_idx] = blue_color
        highlight_atoms.append(atom_idx)

        if rad > 0:
            env = Chem.FindAtomEnvironmentOfRadiusN(mol, rad, atom_idx)
            for b_idx in env:
                # FORCE BONDS TO BE RED
                bond_colors[b_idx] = red_color
                highlight_bonds.append(b_idx)

                # Get atoms connected to this bond
                bond = mol.GetBondWithIdx(b_idx)
                aid1, aid2 = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()

                # If these atoms aren't the center, make them red
                for aid in [aid1, aid2]:
                    highlight_atoms.append(aid)
                    if aid not in atom_colors:  # Don't overwrite the blue center
                        atom_colors[aid] = red_color

    # 3. Draw
    drawer = rdMolDraw2D.MolDraw2DSVG(600, 600)

    # This prevents the "bleeding" of colors
    drawer.drawOptions().prepareMolsBeforeDrawing = True

    drawer.drawOptions().highlightRadius = 0.4
    drawer.DrawMolecule(
        mol,
        highlightAtoms=list(set(highlight_atoms)),
        highlightAtomColors=atom_colors,
        highlightBonds=list(set(highlight_bonds)),
        highlightBondColors=bond_colors,
        legend=f'{i}'
    )
    drawer.FinishDrawing()
    img = drawer.GetDrawingText()

    return replace_circles_with_rectangles(img)

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.Draw import rdMolDraw2D

def visualize_unique_bits_on_molecules(smiles_list, target_bit, radius=2, n_bits=2048, use_chiral=True, molsPerRow=3, subImgSize=(300, 300)):
    """
    Groups molecules by their core bit structure for a specific target_bit
    and visualizes the unique substructures directly on an example molecule
    from each group.
    """
    unique_cores = {}

    # 1. Group by Unique Cores
    for m_idx, smiles in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smiles)
        if not mol:
            continue

        Chem.AssignStereochemistry(mol, force=True, cleanIt=True)
        bit_info = {}
        AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits, bitInfo=bit_info,
                                              useChirality=use_chiral)

        if target_bit in bit_info:
            # Get the core substructure for aggregation
            atomId, rad = bit_info[target_bit][0]
            bitPath = Chem.FindAtomEnvironmentOfRadiusN(mol, rad, atomId)

            # Extract just the core (black part) to generate a grouping key
            core_submol = Chem.PathToSubmol(mol, bitPath)
            core_key = Chem.MolToSmiles(core_submol, isomericSmiles=use_chiral)

            if core_key not in unique_cores:
                unique_cores[core_key] = {
                    'mol': mol,
                    'instances': bit_info[target_bit],
                    'mol_indices': set()
                }
            unique_cores[core_key]['mol_indices'].add(m_idx)

    if not unique_cores:
        return "No occurrences found."

    # 2. Prepare Drawing Data
    mols_to_draw = []
    legends = []
    all_highlight_atoms = []
    all_highlight_bonds = []
    all_atom_colors = []
    all_bond_colors = []

    red_color = (0.204, 1, 0.941, 0.4)
    blue_color = (0.788, 0, 0.329, 0.4)

    for core_key, data in unique_cores.items():
        mol = data['mol']
        mols_to_draw.append(mol)
        legends.append(f"Found in {len(data['mol_indices'])} molecules")

        highlight_atoms = []
        highlight_bonds = []
        atom_colors = {}
        bond_colors = {}

        # Reconstruct highlights over all bit instances in this specific molecule
        for atom_idx, rad in data['instances']:
            # Mark center as Blue
            atom_colors[atom_idx] = blue_color
            highlight_atoms.append(atom_idx)

            if rad > 0:
                env = Chem.FindAtomEnvironmentOfRadiusN(mol, rad, atom_idx)
                for b_idx in env:
                    # Force bonds to be Red
                    bond_colors[b_idx] = red_color
                    highlight_bonds.append(b_idx)

                    # Get atoms connected to this bond
                    bond = mol.GetBondWithIdx(b_idx)
                    aid1, aid2 = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()

                    # If these atoms aren't the center, make them red
                    for aid in [aid1, aid2]:
                        highlight_atoms.append(aid)
                        if aid not in atom_colors:  # Don't overwrite the blue center
                            atom_colors[aid] = red_color

        all_highlight_atoms.append(list(set(highlight_atoms)))
        all_highlight_bonds.append(list(set(highlight_bonds)))
        all_atom_colors.append(atom_colors)
        all_bond_colors.append(bond_colors)

    # 3. Draw Grid Layout
    nRows = len(mols_to_draw) // molsPerRow
    if len(mols_to_draw) % molsPerRow:
        nRows += 1

    mCols = min(len(mols_to_draw), molsPerRow)
    nRows = max(1, nRows) # Ensure at least 1 row exists if only 1 item

    fullSize = (mCols * subImgSize[0], nRows * subImgSize[1])

    drawer = rdMolDraw2D.MolDraw2DSVG(fullSize[0], fullSize[1], subImgSize[0], subImgSize[1])

    # Drawing options styling
    drawOptions = drawer.drawOptions()
    drawOptions.prepareMolsBeforeDrawing = True
    drawOptions.highlightRadius = 0.4

    # Issue drawing command mapping properties atom-by-atom & bond-by-bond
    drawer.DrawMolecules(
        mols_to_draw,
        legends=legends,
        highlightAtoms=all_highlight_atoms,
        highlightAtomColors=all_atom_colors,
        highlightBonds=all_highlight_bonds,
        highlightBondColors=all_bond_colors
    )
    drawer.FinishDrawing()
    img = drawer.GetDrawingText()

    # Apply SVG transformation fix (from initial code block)
    return replace_circles_with_rectangles(img)

In [ ]:
target_bits = [428, 456, 726, 893]

RADIUS = 2
N_BITS = 1024
USE_CHIRALITY = True
OUTPUT_FOLDER = "bit_visualizations"

In [ ]:
import os
import pandas as pd

df = pd.read_csv('../data/herg_data/herg_ecfp_linear.csv')

smiles_lists = df['smiles'].tolist()

In [ ]:
from IPython.core.display import SVG

from rdkit import RDLogger

# Disable RDKit warnings
lg = RDLogger.logger()
lg.setLevel(RDLogger.CRITICAL)

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

for target_bit_id in target_bits:
    print(target_bit_id)
    os.makedirs(os.path.join(OUTPUT_FOLDER, f'ECFP{target_bit_id}'), exist_ok=True)
    img = summarize_bits(smiles_lists, target_bit_id, radius=RADIUS, n_bits=N_BITS,
                   use_chiral=USE_CHIRALITY)
    with open(f'{OUTPUT_FOLDER}/ECFP{target_bit_id}/all_types.svg', 'w') as f:
        f.write(img)
    display(SVG(img))

    img_examples = visualize_unique_bits_on_molecules(smiles_lists, target_bit_id, radius=RADIUS, n_bits=N_BITS,
                   use_chiral=USE_CHIRALITY)
    with open(f'{OUTPUT_FOLDER}/ECFP{target_bit_id}/examples.svg', 'w') as f:
        f.write(img_examples)
    display(SVG(img_examples))